<a href="https://colab.research.google.com/github/wunnakueleon/Machine-Learning/blob/main/Wunna's_ToDo_List_Reality_Check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# To-Do List Reality Check


## Overview

To-Do List Reality Check is a machine learning model that will predict the possibility of a task being finished or not. Personally, I have been very much inclined to multitasking and learning a variety of subjects and topics simultaeneously. Not only that, but also take pleasure in writing unrealistic tasks while at the end of the day, I ended up finishing none of the tasks that I have set.  
  
And thus, To-Do List Reality Check was born, a ML model that will give me a prediction of whether a task is viable or not.   

One-line description: A Reality Check to humble for those who have been delusional with their Absurdity on their potency for the to-do lists ☺️


## Approach



Starting with a base model from Hugging Face: [DistilBert(uncased)](https://huggingface.co/distilbert/distilbert-base-uncased) model.

#### Feature Extraction

Firstly, I will use Feature Extraction for this pre-trained model. Freezing the DistilBert's weights, I will take the embeddings from the frozen layers fromt the model and then train a new Linear layer and then use Sigmoid to produce a prediction of either "Possible" or "Not Possible".

Data Input -> Frozen weights -> embeddings -> Linear Regression ->   Possible/Not Possible

### Dataset

For the dataset for the model to train on, the features will be as following:

- Subject (ML, Math, CSC105, GEN)
- Difficulty level (Easy, Medium, Hard)
- Time taken (30 mins, 1 hr, 3 hrs, 5 hrs)
- Date (Todo lists will fall on the same date)
  
The model will predict the tasks from the same date. And among the todo-list tasks from that day, it will predict whether to shorten the tasks or not by providing prediction for each individual task as 'Possible' or 'Not Possible'.

The dataset will hold ~500 rows.

#### Manual Data Production with Claude Code

Since there's no sufficient record of my past todo lists, I have Claude Code to produce makeshift datasets based on the features of the ones mentioned above.

#### Predicting against my study time

The dataset was produced by prompting Claude to generate data that aligns with the self-study time I have each day: 5 hours.

Easy tasks take 30 minutes or 1 hour, Medium tasks take 3 hours, and Hard tasks take 5 hours. If there is one Easy task and one Hard task on the same date, the model should predict "Not Possible" since the total exceeds 5 hours. However, if there is only one Hard task in a day, it should predict "Possible."

Of course, this problem could simply be solved programmatically by summing up the total hours of all tasks on the same date and checking whether it exceeds 5 hours. Nevertheless, since this is an ML project, the point is to build a model that learns the pattern on its own: predicting "Possible" when the total time doesn't exceed 5 hours and "Not Possible" when it does.

## Model

### Dataset Preview

#### Uploading csv file

In [7]:
from google.colab import files
uploaded = files.upload()  # select task_feasibility_dataset.csv

import pandas as pd
df = pd.read_csv("task_feasibility_dataset.csv")
print(df.shape)
df.head()

Saving task_feasibility_dataset.csv to task_feasibility_dataset (4).csv
(505, 5)


,Subject,Difficulty,Time_Taken_hrs,Date,Possibility
0,GEN,Hard,5.0,2026-04-08,Possible
1,CSC105,Easy,0.5,2026-04-09,Possible
2,Math,Easy,1.0,2026-04-09,Possible
3,ML,Easy,0.5,2026-04-09,Possible
4,ML,Hard,5.0,2026-04-10,Possible


In [8]:
df.tail()

,Subject,Difficulty,Time_Taken_hrs,Date,Possibility
500,CSC105,Easy,0.5,2026-10-30,Not Possible
501,GEN,Medium,3.0,2026-10-30,Not Possible
502,CSC105,Easy,1.0,2026-10-31,Possible
503,ML,Easy,1.0,2026-10-31,Possible
504,Math,Medium,3.0,2026-10-31,Possible


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 505 entries, 0 to 504
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Subject         505 non-null    object 
 1   Difficulty      505 non-null    object 
 2   Time_Taken_hrs  505 non-null    float64
 3   Date            505 non-null    object 
 4   Possibility     505 non-null    object 
dtypes: float64(1), object(4)
memory usage: 19.9+ KB


In [10]:
df.describe()

,Time_Taken_hrs
count,505.000000
mean,1.947525
std,1.504859
min,0.500000
25%,0.500000
50%,1.000000
75%,3.000000
max,5.000000


### Changing each row data  to text

In [11]:
texts = []

def row_to_text(row):
    return f"Subject: {row['Subject']}, Date: {row['Date']}, Difficulty: {row['Difficulty']}, Hours: {row['Time_Taken_hrs']}"

for i, row in df.iterrows():
    texts.append(row_to_text(row))

print(texts[0])
print(texts[1])
print(texts[67])


Subject: GEN, Date: 2026-04-08, Difficulty: Hard, Hours: 5.0
Subject: CSC105, Date: 2026-04-09, Difficulty: Easy, Hours: 0.5
Subject: CSC105, Date: 2026-05-09, Difficulty: Hard, Hours: 5.0


### Labling

In [12]:
labels = df["Possibility"].map({"Possible": 1, "Not Possible": 0})
print(labels.value_counts())

Possibility
1    283
0    222
Name: count, dtype: int64


### Training and Testing Data Split

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, stratify=labels, random_state=67)

### Loading DistilBert

#### Installation & Importing Classes for DistilBert

In [14]:
import torch
import numpy as np
from transformers import DistilBertTokenizer, DistilBertModel

#### Loading Tokenizer and Model

In [15]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = DistilBertModel.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#### Freezing all of the pre-trained model's weights

In [16]:
model.eval()
for param in model.parameters():
    param.requires_grad = False

#### Moving to GPU

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Model loaded on: {device}")

Model loaded on: cuda


### Extract Embeddings

In [18]:
# Training row by row without grouping the tasks with the date

# def extract_embeddings(texts, batch_size=16):
#     all_embeddings = []
#     for i in range(0, len(texts), batch_size):
#         batch = texts[i : i + batch_size]
#         encoded = tokenizer(
#             batch,
#             padding=True,
#             truncation=True,
#             max_length=128,
#             return_tensors="pt"
#         ).to(device)
#         with torch.no_grad():
#             output = model(**encoded)
#         cls_emb = output.last_hidden_state[:, 0, :]
#         all_embeddings.append(cls_emb.cpu().numpy())
#     return np.vstack(all_embeddings)




# Training by grouping tasks from the same day and train group by group

grouped = df.groupby('Date').apply(
    lambda g: pd.Series({
        'text': ' | '.join(
            f"{row['Subject']} {row['Difficulty']} {row['Time_Taken_hrs']}hrs"
            for _, row in g.iterrows()
        ),
        'Possibility': g['Possibility'].iloc[0]
    })
).reset_index()

# Example: "ML Easy 1hrs | Math Hard 5hrs" → Not Possible
print(grouped[['text', 'Possibility']].head())


X = grouped['text'].tolist()
y = grouped['Possibility'].map({'Possible': 1, 'Not Possible': 0}).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


def extract_embeddings(texts, batch_size=16):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"

        ).to(device)
        with torch.no_grad():
            output = model(**encoded)
        cls_emb = output.last_hidden_state[:, 0, :]
        all_embeddings.append(cls_emb.cpu().numpy())
    return np.vstack(all_embeddings)


print("Extracting train embeddings...")
X_train_emb = extract_embeddings(X_train)

print("Extracting test embeddings...")
X_test_emb = extract_embeddings(X_test)

print(X_train_emb.shape)

                                                text   Possibility
0                                    GEN Hard 5.0hrs      Possible
1  CSC105 Easy 0.5hrs | Math Easy 1.0hrs | ML Eas...      Possible
2                                     ML Hard 5.0hrs      Possible
3  ML Medium 3.0hrs | Math Easy 1.0hrs | CSC105 E...      Possible
4             GEN Hard 5.0hrs | CSC105 Medium 3.0hrs  Not Possible
Extracting train embeddings...


/tmp/ipykernel_436/3185530256.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = df.groupby('Date').apply(


Extracting test embeddings...
(165, 768)


### Training

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_emb, y_train)

y_pred = clf.predict(X_test_emb)
print(classification_report(y_test, y_pred, target_names=["Not Possible", "Possible"]))

              precision    recall  f1-score   support

Not Possible       1.00      0.96      0.98        24
    Possible       0.95      1.00      0.97        18

    accuracy                           0.98        42
   macro avg       0.97      0.98      0.98        42
weighted avg       0.98      0.98      0.98        42



## Experimentation

Machine Learning is iterative. These are the experimentations of the project.

| Experiment | Change | Accuracy | Notes |
|------------|--------|----------|-------|
| Exp 1      | Each row trained individually without grouping tasks by date |   72%    | Model fails to predict feasibility when multiple tasks fall on the same day, with 5-hour self-study tasks |  
| Exp 2     | Group the individual tasks from the same date | 98% | Model succeeds to grasp the pattern: predict Possibility of the tasks done based on 5-hour self-study time per day